# Dialogue Routing and Management

## Import

In [38]:
# Standard libraries
import re
import string
import random
import pickle
import os
import json
from typing import Optional, Dict
import random
import logging
from enum import Enum
from datetime import datetime, timedelta
import json
from random import choice

# Data processing
import pandas as pd
import numpy as np
from sklearn.utils import shuffle
from sklearn.model_selection import train_test_split
import onnxruntime as ort

# Visualization
import matplotlib.pyplot as plt

# Text preprocessing & NLP
from sklearn.feature_extraction.text import TfidfVectorizer
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Machine Learning
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_recall_fscore_support, roc_auc_score

# Deep Learning - TensorFlow / Keras
import tensorflow as tf
from tensorflow.keras.layers import Lambda
import tensorflow as tf
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout, Concatenate
from tensorflow.keras.optimizers import Adam
from keras.saving import register_keras_serializable
from tensorflow.keras.layers import (
    Input, Embedding, SpatialDropout1D, Bidirectional, LSTM,
    Dense, Dropout, LayerNormalization, Lambda, Concatenate,BatchNormalization
)
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.regularizers import l2

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import keras
keras.config.enable_unsafe_deserialization()


In [39]:
tokenizer = AutoTokenizer.from_pretrained("../generated/tokenizer_intent_classifier_indobert")
onnx_model_path = "../generated/intent_classifier.onnx"
ort_session = ort.InferenceSession(onnx_model_path)


In [40]:
@register_keras_serializable(package="Custom")
def triplet_loss(y_true, y_pred, margin=0.5):
    
    # Bagi menjadi tiga embedding sepanjang dim-1
    anchor, positive, negative = tf.split(y_pred, num_or_size_splits=3, axis=1)

    # Hitung squared Euclidean distance
    pos_dist = tf.reduce_sum(tf.square(anchor - positive), axis=1)
    neg_dist = tf.reduce_sum(tf.square(anchor - negative), axis=1)

    # Triplet loss
    basic_loss = pos_dist - neg_dist + margin
    loss = tf.reduce_mean(tf.maximum(basic_loss, 0.0))
    return loss

@register_keras_serializable(package="Custom")
def l2_normalize(t):
    return tf.math.l2_normalize(t, axis=1)

In [41]:
retriever_model = tf.keras.models.load_model(
    "../generated/siamese_model.keras",
    custom_objects={
        "triplet_loss": triplet_loss,
        "tf": tf,  # tambahkan ini
        "l2_normalize": l2_normalize
    }
)
encoder = retriever_model.get_layer("shared_encoder")

In [42]:
INTENTS = {
    0: "ask_first_aid_solution",
    1: "ask_possible_cause",
    2: "casual_greeting",
    3: "fallback",
    4: "goodbye",
    5: "report_noise_or_smell",
}



In [43]:
def preprocess(text):
    return np.array([ord(c) for c in text.lower() if c.isalnum()])

In [44]:
def pad_sequence(seq, maxlen=30):
    # Pastikan input adalah list, bukan np.array
    if isinstance(seq, np.ndarray):
        seq = seq.tolist()
    seq = seq[:maxlen]                  # truncate kalau terlalu panjang
    seq += [0] * (maxlen - len(seq))   # padding dengan 0
    return np.array(seq)


In [45]:
class DialogManager:
    def __init__(self):
        self.sessions = {}  # user_id -> state dict

    def get_state(self, user_id):
        return self.sessions.get(user_id, {"last_intent": None, "waiting_for": None, "slots": {}})

    def update_state(self, user_id, intent, slots=None):
        state = self.get_state(user_id)
        state["last_intent"] = intent
        if slots:
            state["slots"].update(slots)
        self.sessions[user_id] = state

    def clear_state(self, user_id):
        if user_id in self.sessions:
            del self.sessions[user_id]

In [46]:
def softmax(x, axis=None):
    e_x = np.exp(x - np.max(x, axis=axis, keepdims=True))
    return e_x / np.sum(e_x, axis=axis, keepdims=True)

In [47]:
def predict_intent(text: str) -> tuple[str, float]:
    inputs = tokenizer(text, return_tensors="np", padding=True, truncation=True)
    input_ids = inputs["input_ids"].astype(np.int64)
    attention_mask = inputs["attention_mask"].astype(np.int64)

    ort_inputs = {
        "input_ids": input_ids,
        "attention_mask": attention_mask
    }

    logits = ort_session.run(["logits"], ort_inputs)[0]
    print(logits.shape)  
    probs = softmax(logits, axis=1)
    predicted_class = np.argmax(probs, axis=1)[0]
    confidence = float(np.max(probs))

    intent_labels = [
        "ask_first_aid_solution",
        "ask_possible_couse",
        "casual_greeting",
        "fallback",
        "goodbye",
        "repost_noise_or_smell",
    ]

    
    return intent_labels[predicted_class], confidence


In [48]:
def get_embedding(text):
    seq = preprocess(text)
    seq = pad_sequence(seq, maxlen=30)
    seq = np.expand_dims(seq, axis=0)
    emb = encoder.predict(seq)
    return emb


In [49]:
def fallback_intent(text, known_intents_emb):
    text_emb = get_embedding(text)
    similarities = np.dot(known_intents_emb, text_emb.T).flatten()
    best_idx = np.argmax(similarities)
    if similarities[best_idx] > 0.7:  # threshold fallback
        return INTENTS[best_idx]
    return "fallback"

In [50]:
def handle_intent(user_id, intent, dialog_manager, user_input=None):
    state = dialog_manager.get_state(user_id)

    if intent == "ask_first_aid_solution":
        return "Silakan ceritakan keluhan kendaraan Anda, saya akan coba bantu memberikan pertolongan pertama yang sesuai."

    if intent == "ask_possible_cause":
        return "Ceritakan gejala yang Anda alami, saya akan bantu prediksi kemungkinan penyebabnya."

    if intent == "casual_greeting":
        dialog_manager.clear_state(user_id)
        return "Hai! Saya siap membantu Anda terkait masalah kendaraan. Silakan ceritakan apa yang Anda alami."

    if intent == "fallback":
        dialog_manager.clear_state(user_id)
        return "Maaf, saya kurang paham maksud Anda. Bisa dijelaskan kembali dengan kata lain?"

    if intent == "goodbye":
        dialog_manager.update_state(user_id, intent)
        return "Terima kasih telah menggunakan layanan kami. Semoga kendaraan Anda segera pulih!"

    if intent == "report_noise_or_smell":
        dialog_manager.update_state(user_id, intent)
        return "Baik, suara atau bau yang tidak biasa bisa mengindikasikan masalah. Bisa dijelaskan lebih detail gejalanya?"

    return "Mohon maaf, fitur ini belum tersedia."



In [51]:
dialog_manager = DialogManager()

def chatbot_response(user_id, user_input):
    intent, confidence = predict_intent(user_input)
    if confidence < 0.6:
        intent = fallback_intent(user_input, KNOWN_INTENTS_EMB)
    return handle_intent(user_id, intent, dialog_manager, user_input)

In [56]:
user_id = "user123"
KNOWN_INTENTS_EMB = np.array([get_embedding(intent_name) for intent_name in INTENTS.values()])

print(chatbot_response(user_id, "Saya mendengar suara aneh dari mesin mobil saya."))  # Contoh input       

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 42ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 43ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 46ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step
(1, 6)
Mohon maaf, fitur ini belum tersedia.
